### This notebook was used to make all of my data requests from NBA.com

In [3]:
import pandas as pd
import requests
import json
import time
import random
from nba_api.stats.library.http import NBAStatsHTTP
from nba_api.stats.endpoints import commonteamroster, shotchartdetail, leaguegamefinder, playbyplayv3

In [ ]:
# 1610612760: "Thunder",
#    1610612759: "Spurs",
#    1610612765: "Pistons",
#    1610612738: "Celtics",
#    1610612752: "Knicks",
#    1610612762: "Jazz",
#    1610612758: "Kings",
#    1610612751: "Nets",
#    1610612764: "Wizards",
#    1610612754: "Pacers"
# 1610612739: "Cavaliers",
# 1610612745: "Rockets"
# 1610612740: "Pelicans",
#    1610612755: "76ers",
#    1610612766: "Hornets"
# 1610612743: "Nuggets",
# 1610612757: "Trail Blazers",
# 1610612746: "Clippers"
# 1610612750: "Timberwolves"
# 1610612753: "Magic"
#    1610612756: "Suns",
#    1610612763: "Grizzlies",
#    1610612744: "Warriors",
#    1610612748: "Heat",
#    1610612742: "Mavericks"
# 1610612737: "Hawks"
# 1610612749: "Bucks",
# 1610612761: "Raptors",
# 1610612747: "Lakers",
#    1610612741: "Bulls"


In [ ]:
# Code to pull shot charts.
# TEAM_IDS were switched in and out as I made requests for different seasons.

TEAM_IDS = [
    1610612759,
    1610612760,
    1610612746,
    1610612754,
    1610612745]

SEASON = '2012-13'

TEAM_NAMES = {
    1610612759: "Spurs",
    1610612760: "Thunder",
    1610612746: "Clippers",
    1610612754: "Pacers",
    1610612745: "Rockets"
}

for TEAM_ID in TEAM_IDS:
    team_name = TEAM_NAMES.get(TEAM_ID, "Unknown")
    print(f"\nProcessing {team_name} ({TEAM_ID})...")

    # -----------------------------
    # Step 1: Get roster (WITH RETRY)
    # -----------------------------
    roster_success = False

    for attempt in range(3):
        try:
            print(f"Fetching roster | attempt {attempt+1}")

            roster = commonteamroster.CommonTeamRoster(
                team_id=TEAM_ID,
                season=SEASON,
                timeout=60
            )

            players_df = roster.get_data_frames()[0]
            player_ids = players_df['PLAYER_ID'].tolist()

            roster_success = True
            break

        except Exception as e:
            print(f"Roster error: {e}")
            time.sleep(5)

    if not roster_success:
        print(f"Skipping team {team_name} (roster failed)")
        continue

    print(f"Found {len(player_ids)} players")

    # -----------------------------
    # Step 2: Pull shot data
    # -----------------------------
    team_shots = []

    for pid in player_ids:
        success = False

        for attempt in range(3):
            try:
                print(f"{team_name} | Player {pid} | attempt {attempt+1}")

                response = shotchartdetail.ShotChartDetail(
                    team_id=TEAM_ID,
                    player_id=pid,
                    season_nullable=SEASON,
                    season_type_all_star='Regular Season',
                    context_measure_simple='FGA',
                    timeout=60
                )

                df = response.get_data_frames()[0]

                df['TEAM_ID'] = TEAM_ID
                df['TEAM_NAME'] = team_name
                df['PLAYER_ID'] = pid

                team_shots.append(df)

                success = True
                break

            except Exception as e:
                print(f"Error: {e}")
                time.sleep(3)

        if not success:
            print(f"Skipping player {pid}")

        time.sleep(2.5)

    # -----------------------------
    # Step 3: Save per team
    # -----------------------------
    if len(team_shots) == 0:
        print(f"No data for {team_name}, skipping save")
        continue

    team_df = pd.concat(team_shots, ignore_index=True)

    filename = f"{team_name.replace(' ', '_').lower()}_201314_shot_chart.csv"
    team_df.to_csv(filename, index=False)

    print(f"\nSaved {team_name} data to {filename}")

    # Extra delay between teams
    time.sleep(10)

In [ ]:
# Code to pull free throw datasets

# Reset session (IMPORTANT to avoid blocking)
NBAStatsHTTP()._session = None


# ---------------------------
# Step 1: Get all games
# ---------------------------
print("Fetching game list...")

games = leaguegamefinder.LeagueGameFinder(
    season_nullable="2013-14",
    season_type_nullable="Regular Season",
    player_or_team_abbreviation='T',
    timeout=60
)

games_df = games.get_data_frames()[0]
game_ids = games_df['GAME_ID'].unique()

# Run only on first 600 games
game_ids = game_ids[800:]

print("Total games to process:", len(game_ids))


# ---------------------------
# Step 2: Loop through games
# ---------------------------
all_free_throws = []

for i, game_id in enumerate(game_ids):

    print(f"Processing {i+1}/{len(game_ids)}: Game {game_id}")

    retries = 3

    for attempt in range(retries):

        try:
            pbp = playbyplayv3.PlayByPlayV3(
                game_id=game_id,
                start_period=1,
                end_period=10,
                timeout=60
            )

            df = pbp.get_data_frames()[0]

            # Filter free throws
            ft = df[df["actionType"] == "Free Throw"].copy()

            if ft.empty:
                break

            # Add useful columns
            ft["GAME_ID"] = game_id
            ft["FT_ATTEMPT"] = 1

            # Correct FT Made Logic
            ft["FT_MADE"] = ~ft["description"].str.upper().str.startswith("MISS")

            # Handle missing columns safely
            if "playerNameI" not in ft.columns:
                ft["playerNameI"] = None

            if "teamTricode" not in ft.columns:
                ft["teamTricode"] = None

            # Select final columns
            ft = ft[
                [
                    "GAME_ID",
                    "period",
                    "clock",
                    "playerNameI",
                    "teamTricode",
                    "FT_ATTEMPT",
                    "FT_MADE",
                    "subType",
                    "description"
                ]
            ]

            all_free_throws.append(ft)

            # Slightly smarter sleep
            time.sleep(1.1 + random.random() * 0.7)

            break


        except Exception as e:

            print(f"Retry {attempt+1} failed for {game_id}: {e}")
            time.sleep(2 + attempt)

            # If still failing → cooldown instead of skipping
            if attempt == retries - 1:
                print("Heavy throttling detected... cooling down")
                time.sleep(20)
                NBAStatsHTTP()._session = None


    # ---------------------------
    # Save backup every 50 games
    # ---------------------------
    if i % 50 == 0 and len(all_free_throws) > 0:
        temp_df = pd.concat(all_free_throws, ignore_index=True)
        temp_df.to_csv("2020_21_nba_free_throw_backup.csv", index=False)
        print("Backup saved...")


    # ---------------------------
    # Cooldown every 75 games
    # ---------------------------
    if i % 75 == 0 and i != 0:
        print("Cooling down to avoid throttling...")
        time.sleep(15)
        NBAStatsHTTP()._session = None


# ---------------------------
# Step 3: Combine All Games
# ---------------------------
if all_free_throws:

    free_throws_df = pd.concat(all_free_throws, ignore_index=True)

    print("Total free throws extracted:", len(free_throws_df))

    # ---------------------------
    # Step 4: Save Final CSV
    # ---------------------------
    free_throws_df.to_csv("nba_2013_14_free_throws3.csv", index=False)

    print("Saved to nba_2013_14_free_throws3.csv")

else:
    print("No free throws extracted.")

In [73]:
# Re-combine free same seasons free throws sets
ft1 = pd.read_csv('nba_2013_14_free_throws.csv')
ft2 = pd.read_csv('nba_2013_14_free_throws2.csv')
ft3 = pd.read_csv('nba_2013_14_free_throws.csv')

ft_combined = pd.concat([ft1, ft2, ft3], ignore_index=True)

# Change FT_MADE to 0s or 1s to match other shot data.
ft_combined["FT_MADE"] = ft_combined["FT_MADE"].astype(int)

In [71]:
# Save data to csv
ft_combined.to_csv('nba_2013_14_free_throws_combined.csv', index=False)